<font color=#009afa size=50>DAT-6002 - Fundamentals of Artificial Intelligence</font><br>
<font size=50>WRIT1 Part 2 - Neural Networks</font><br>
<font size=20 color=#00c382>From Scratch MLP - Bank Data</font><br>

|**Assessment Code:** |WRIT1|
|---|---|
|**Author:**|Tom Rowan|
|**Course:**|DAT6002_S2_25|
|**Course Tutor:**|Imtiaz Hussain Khan, PhD|


**Coding Syntax Note: _variable_name --> temporary / locally used variable, discarded after use.**

https://archive.ics.uci.edu/dataset/222/bank+marketing

---

# Initialise Notebook

In [374]:
# Install Requirements Using Pip
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


### Import Python Libraries

In [375]:
# Standard Libraries
import sys
import time
import json
import random
from datetime import datetime

# Pandas and numpy
import pandas as pd
import numpy as np

# Matplot Lib
import matplotlib.pyplot as plt
from matplotlib import colormaps
import matplotlib.colors as mcolors
from matplotlib.pyplot import figure
import matplotlib.ticker as mticker
import matplotlib.animation as animation
%matplotlib inline

# Seaborn Graphs
import seaborn as sns

# Plotly
import plotly.express as px
import plotly.graph_objects as go

# Tabulate
from tabulate import tabulate

# Colorama
from colorama import Fore, Back, Style

# Requests
import requests

# Pycountry 
import pycountry

# GeoPy
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# Cartopy Maps
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Geopandas
import geopandas as gpd

# IPython Display 
from IPython.display import HTML

# Do not print warnings!
import warnings
warnings.filterwarnings('ignore')

### Create a Timestamp

In [376]:
# Generate Timestamp
today = pd.Timestamp(datetime.today())
now_date_timestamp = str(today.strftime("%Y%m%d"))
now_datetime_timestamp = str(today.strftime("%Y%m%d-%H%M"))

### Functions for Pretty Output

print_output - Print coloured Output

In [377]:
# Function to print coloured output
def print_output (str1, str2, delim=':', colour='yellow'):
    if colour.upper() in dir(Fore):
        colour = colour.upper()
    else:
        colour = 'YELLOW'
    print (f'{getattr(Fore, colour)}{str(str1)}{delim}{Style.RESET_ALL} {str(str2)}')

### Check for Internet Access

In [378]:
_urls_to_check = ['http://connectivitycheck.gstatic.com/generate_204',
                  'https://www.github.com',
                  'https://datascience.daemonchild.com']

Set this to False if you have a local dataset

In [379]:
internet_required = False

In [380]:
# Function to Safely Check for Internet Connectivity
def check_internet (url_list):
    online = False
    for _url in url_list:
        try:
            response = requests.get(_url, timeout=3)
            if response.status_code == 200:
                online = True
                break
        except requests.RequestException:
            output_string = f"Check Internet failed for {_url}"
            print_output ('[FAIL]',output_string, colour='red', delim='')
    return online

# Perform the checks
if check_internet (_urls_to_check ):
    print_output ('[INTERNET]','Internet Connectivity Detected. Using Internet resources.',colour='green', delim='')
    online = True

else:
    print ()
    print_output ('[INTERNET]','No Internet Connectivity.', colour='red', delim='')
    online = False


# Stop running if Internet is required
if online == False & internet_required == True:
    sys.exit('Stopping notebook run here.')    


[INTERNET] Internet Connectivity Detected. Using Internet resources.


---
# Load and Explore the Dataset

In [381]:
if online:
    # Fetch the dataset from the Internet
    df = pd.read_csv('bank-full.csv', delimiter=';')
else:
    # Load the dataset from local file
    df = pd.read_csv('bank-full.csv', delimiter=';')

### Explore the dataset

In [382]:
print (df.columns)
print ("")
print (df.shape)

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='str')

(45211, 17)


In [383]:
df.sample(20)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
20534,34,housemaid,married,tertiary,no,897,no,no,cellular,12,aug,197,4,-1,0,unknown,no
13461,40,admin.,married,secondary,no,0,yes,yes,cellular,9,jul,145,1,-1,0,unknown,no
3024,32,technician,single,unknown,no,10600,no,no,unknown,14,may,485,2,-1,0,unknown,no
15572,26,services,single,secondary,no,3,no,yes,cellular,21,jul,140,4,-1,0,unknown,no
25175,47,admin.,single,secondary,no,19,yes,yes,cellular,18,nov,90,2,123,1,failure,no
24516,56,management,married,tertiary,no,0,no,no,cellular,17,nov,194,1,-1,0,unknown,no
21711,34,technician,single,secondary,no,179,no,no,cellular,19,aug,294,3,-1,0,unknown,no
4457,34,services,married,primary,no,1042,yes,no,unknown,20,may,167,1,-1,0,unknown,no
6651,31,blue-collar,married,primary,no,-454,yes,yes,unknown,28,may,861,2,-1,0,unknown,no
20431,31,management,single,tertiary,no,123,no,no,cellular,12,aug,82,2,-1,0,unknown,no


In [384]:
df.dropna(inplace=True)
df.shape

(45211, 17)

## Conversion and Encoding Ready for the MLP

xscaled​=max(x)−min(x)x−min(x)​

In [385]:
numerical_cols = df.select_dtypes(include=['number']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Binary columns that should not be one-hot encoded
binary_columns = ['default', 'housing', 'loan', 'y']

# Columns to exclude from the system
excluded_features = ['day', 'campaign', 'pdays', 'previous', 'y']

numerical_to_normalize = [col for col in numerical_cols if col not in binary_columns]
categorical_to_normalize = [col for col in categorical_cols if col not in binary_columns]

In [386]:
print (numerical_cols)
print (numerical_to_normalize)
print (categorical_cols)
print (categorical_to_normalize)
print (binary_columns)

['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'y']
['job', 'marital', 'education', 'contact', 'month', 'poutcome']
['default', 'housing', 'loan', 'y']


In [387]:
def max_min_normalize (df, column_name):
    max_value = df[column_name].max()
    min_value = df[column_name].min()
    df[column_name] = (df[column_name] - min_value) / (max_value - min_value)
    return df

def label_encode(df, column_name):
    # Get unique values in the column
    unique_values = df[column_name].unique()
    
    # Create a mapping of unique values to integers
    mapping = {value: i for i, value in enumerate(unique_values)}
    
    # Apply the mapping to the column
    df[column_name] = df[column_name].map(mapping).astype(float)
    return df

In [388]:


# Clean up the binary columns to be 0 and 1
for col in binary_columns:
    df[col] = df[col].map({'yes': 1.0, 'no': 0.0}).astype(float)

for col in categorical_to_normalize:
    df = label_encode(df, col)
    df = max_min_normalize(df, col)
    
for col in numerical_to_normalize:
    df = max_min_normalize(df, col)




In [397]:
df.sample(10)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
32475,0.337662,0.727273,0.5,0.333333,0.0,0.073938,1.0,0.0,1.0,0.533333,0.909091,0.022163,0.032258,0.000000,0.000000,0.000000,0.0
1269,0.506494,0.000000,0.0,0.000000,0.0,0.084978,1.0,0.0,0.0,0.233333,0.000000,0.068930,0.016129,0.000000,0.000000,0.000000,0.0
27305,0.194805,0.636364,0.5,0.333333,0.0,0.102482,0.0,1.0,0.5,0.666667,0.454545,0.077471,0.000000,0.145642,0.003636,0.333333,0.0
10439,0.363636,0.000000,1.0,0.000000,0.0,0.078932,0.0,1.0,0.0,0.366667,0.090909,0.038634,0.016129,0.000000,0.000000,0.000000,0.0
31337,0.376623,0.272727,0.0,0.333333,0.0,0.074964,1.0,1.0,0.5,0.400000,0.818182,0.625458,0.000000,0.000000,0.000000,0.000000,1.0
17720,0.311688,0.181818,0.0,0.000000,0.0,0.080602,0.0,0.0,0.5,0.933333,0.181818,0.036600,0.145161,0.000000,0.000000,0.000000,0.0
3892,0.324675,0.636364,0.0,1.000000,0.0,0.080266,1.0,0.0,0.0,0.500000,0.000000,0.022773,0.000000,0.000000,0.000000,0.000000,0.0
9131,0.350649,0.272727,0.0,1.000000,0.0,0.083943,0.0,0.0,0.0,0.133333,0.090909,0.025824,0.032258,0.000000,0.000000,0.000000,0.0
17493,0.298701,0.090909,0.0,0.333333,0.0,0.078614,0.0,0.0,0.5,0.900000,0.181818,0.034364,0.080645,0.000000,0.000000,0.000000,0.0
3847,0.077922,0.636364,0.5,0.333333,0.0,0.072803,0.0,0.0,0.0,0.500000,0.000000,0.056934,0.000000,0.000000,0.000000,0.000000,0.0


This from Gemini - why? Need to understand this and reimplement 

In [390]:
sys.exit('Stopping notebook run here.')    

SystemExit: Stopping notebook run here.

---

# Data Cleaning


...do stuff...

### Write out the cleaned dataset for further use



In [ ]:
# Create filenames, one timestamped, one 'latest'
filepath = "."
filename = f'{filepath}/bank_full_cleaned_{now_date_timestamp}.csv'
latest_filename = f'{filepath}/bank_full_cleaned_latest.csv'

# Write out CSV files
df.to_csv(filename, index=False)
df.to_csv(latest_filename, index=False)